# Módulo 2 · 05 · 30 semillas por red final

Cada red final se entrena con las semillas 42 a 71 (hasta 30 épocas, parada temprana con paciencia 5 sobre la pérdida de validación). Se guarda el modelo de la semilla con menor pérdida de validación.

In [ ]:
# parameters
DEMO = False
EJECUTAR = False

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd().resolve().parents[1]
sys.path[:0] = [str(RAIZ), str(RAIZ / "modulo2_distraccion")]

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common.seeds import fijar_semilla
import m2lib, m2exp

fijar_semilla()
cfg = m2exp.config(demo=DEMO)
CLASES = cfg["datos"]["clases"]
ETIQ = [m2lib.NOMBRES[c] for c in CLASES]
FIG, RES = m2lib.FIGURAS, m2lib.RESULTADOS
FIG.mkdir(exist_ok=True); RES.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight"})

In [ ]:
s = "_demo" if DEMO else ""
sem, info = {}, {}
for fam in ["cnn", "transfer"]:
    if EJECUTAR or not (RES / f"semillas_{fam}{s}.csv").exists():
        m2exp.semillas(cfg, fam)
    sem[fam] = pd.read_csv(RES / f"semillas_{fam}{s}.csv")
    info[fam] = json.loads((RES / f"semillas_{fam}{s}.json").read_text())
log = json.loads((RES / f"logistica{s}.json").read_text())
filas = []
for nombre, d in [("Red desde cero", sem["cnn"]), ("Preentrenada + ajuste fino", sem["transfer"])]:
    for m in ["exactitud", "f1_macro"]:
        filas.append({"modelo": nombre, "métrica": m, "media": d[m].mean(), "desv": d[m].std(), "mejor": d[m].max(), "peor": d[m].min()})
for m in ["exactitud", "f1_macro"]:
    filas.append({"modelo": "Regresión logística", "métrica": m, "media": log[m], "desv": 0.0, "mejor": log[m], "peor": log[m]})
resumen = pd.DataFrame(filas).round(4)
resumen.to_csv(RES / "resumen_semillas.csv", index=False)
resumen

In [ ]:
comp = pd.DataFrame({
    "modelo": ["Regresión logística", "Red desde cero", "Preentrenada + ajuste fino"],
    "parámetros entrenables": [log["parametros"], info["cnn"]["parametros"], info["transfer"]["parametros"]],
    "configuraciones probadas": [log["configuraciones"], len(pd.read_csv(RES / f"busqueda_cnn{s}.csv")), len(pd.read_csv(RES / f"busqueda_transfer{s}.csv"))],
    "minutos de GPU (búsqueda + semillas)": [log["segundos"] / 60,
        (pd.read_csv(RES / f"busqueda_cnn{s}.csv")["segundos"].sum() + sem["cnn"]["segundos"].sum()) / 60,
        (pd.read_csv(RES / f"busqueda_transfer{s}.csv")["segundos"].sum() + sem["transfer"]["segundos"].sum()) / 60],
})
comp.to_csv(RES / "complejidad.csv", index=False)
comp.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
datos_box = [sem["cnn"]["f1_macro"], sem["transfer"]["f1_macro"]]
ax.boxplot(datos_box, tick_labels=["Red desde cero", "Preentrenada\n+ ajuste fino"])
ax.axhline(log["f1_macro"], color="C3", ls="--", label="Regresión logística (determinista)")
ax.set_ylabel("F1 macro en prueba"); ax.legend(loc="lower right")
fig.savefig(FIG / "comparacion_semillas.png")

In [ ]:
hist = {fam: pd.read_csv(RES / f"historial_{fam}{s}.csv") for fam in ["cnn", "transfer"]}
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, fam in zip(axs, ["cnn", "transfer"]):
    h = hist[fam][hist[fam]["semilla"] == info[fam]["semilla_guardada"]]
    ax.plot(h["epoca"], h["perdida_ent"], label="entrenamiento")
    ax.plot(h["epoca"], h["perdida_val"], label="validación")
    mejor = h.loc[h["perdida_val"].idxmin(), "epoca"]
    ax.axvline(mejor, color="k", ls=":", label=f"pesos guardados (época {mejor})")
    ax.axvline(h["epoca"].max(), color="C3", ls="--", label=f"parada temprana (época {h['epoca'].max()})")
    ax.set_title({"cnn": "Red desde cero", "transfer": "Preentrenada + ajuste fino"}[fam] + f" · semilla {info[fam]['semilla_guardada']}")
    ax.set_xlabel("época"); ax.set_ylabel("entropía cruzada"); ax.legend()
fig.savefig(FIG / "curvas_aprendizaje.png")

## Arquitecturas

In [ ]:
for fam in ["cnn", "transfer"]:
    red, d = m2exp.cargar_modelo(fam) if not DEMO else (m2lib.construir(*m2exp.hp_final(cfg, fam), len(CLASES)), None)
    print(fam, info[fam]["tipo"], info[fam]["hp"], "parámetros entrenables:", m2lib.parametros_entrenables(red))
print(m2lib.construir("cnn", cfg["cnn"], len(CLASES)))